## Topic 5: Qdrant's Model — Collections, Points, Payloads

### Concept, Intuition, Why It Exists

- Before writing any upsert or search code, it is worth being precise about what Qdrant's data model actually is — because the three terms (collection, point, payload) map directly onto concepts from earlier in this project, and understanding that mapping makes every later topic cleaner.
- A **collection** is a named group of points that all share the same vector size and distance metric. It is the closest equivalent to a table in SQL — a container for a specific kind of data, with a fixed schema for the vector dimension. Unlike a SQL table, the payload schema is not fixed — different points in the same collection can have different payload fields.
- A **point** is one entry in a collection — the atomic unit of storage. Every point has three things: a unique ID (integer or UUID), a vector (the embedding), and a payload (a dict of metadata). This maps directly onto this project's Document shape: `page_content` gets embedded to produce the vector, `metadata` becomes the payload, and a generated ID ties them together.
- A **payload** is the metadata dict attached to a point. It can hold any JSON-serializable fields — strings, numbers, booleans, lists, nested dicts. This is where `source`, `doc_type`, `page`, `chunk_index`, and any other metadata from the ingestion chapter live inside Qdrant. Payloads are what make filtering useful — you filter on payload fields, not on vector content.
- Together: a **collection** holds many **points**, each **point** has a vector for similarity search and a payload for filtering and citation.

### Internal Working

- **Collection creation** requires specifying vector size and distance metric upfront — these cannot be changed after creation without deleting and recreating the collection. Everything else (payload schema, HNSW parameters) can be adjusted or extended later.
- **Point IDs** must be unique within a collection. Qdrant accepts both unsigned integers and UUIDs. Choosing a deterministic ID (e.g. a hash of source path + chunk index) instead of a random one makes upsert idempotent — upserting the same point twice updates it in place rather than creating a duplicate. This is the vector-store equivalent of the incremental ingestion discipline from the ingestion chapter.
- **Upsert semantics**: Qdrant's `upsert` operation inserts a point if its ID doesn't exist, or replaces it entirely if the ID already exists. This is the correct primitive for incremental ingestion — re-running ingest for a changed file simply re-upserts the same IDs with updated vectors and payloads, with no manual delete-then-insert needed.
- **Payload indexing**: by default, payload fields are stored but not indexed for filtering. Adding a payload index on a specific field (e.g. `doc_type`) makes filtered searches on that field significantly faster at large scale. Qdrant supports keyword, integer, float, and geo payload indexes.

### How It's Implemented In This Project

- The mapping from this project's Document shape to Qdrant's data model is direct:
- `Document["page_content"]` gets embedded and becomes `point.vector`
- `Document["page_content"]` is also stored as `point.payload["text"]` so retrieved results are self-contained and don't need a secondary lookup back to source files
- `Document["metadata"]["source"]` becomes `point.payload["source"]`
- `Document["metadata"]["doc_type"]` becomes `point.payload["doc_type"]`
- `Document["metadata"]["page"]` becomes `point.payload["page"]`
- `Document["metadata"]["chunk_index"]` becomes `point.payload["chunk_index"]`
- The point ID is generated as a deterministic integer from a hash of `source + chunk_index`, making upserts idempotent — re-ingesting a changed file updates existing points rather than creating duplicates alongside them.

### Real-World Issues, Edge Cases, Debugging

- **Payload schema flexibility is a double-edged sword**: because Qdrant doesn't enforce a fixed payload schema, one loader using `"doc_type"` and another using `"document_type"` for the same concept silently produces points with inconsistent payload keys — filters on `"doc_type"` miss the points that used `"document_type"`, with no error thrown anywhere. Schema consistency must be enforced at the application layer.
- **Random vs. deterministic point IDs**: using random IDs means re-ingesting a changed file creates new points alongside the old ones — the old points remain with stale vectors and payloads, and both appear in search results. Deterministic IDs ensure re-ingest replaces rather than duplicates.
- **Payload field not indexed**: filtering on a payload field with no payload index forces Qdrant to scan all payload records during search. At small scale this is invisible. At large scale it degrades filtered search latency significantly.
- **Points count vs. indexed vectors count**: immediately after a large batch upsert, Qdrant's `points_count` reflects the new total but the HNSW index may still be updating in the background. Searches during this window may miss recently upserted points.

### Design Decisions & Trade-offs

- Storing text in the payload alongside metadata, rather than keeping a separate lookup table: adds storage cost (text is stored twice — once as the source of the embedding, once in the payload) in exchange for self-contained search results. At this project's corpus size the duplication is negligible and the convenience is worth it.
- Deterministic hash-based IDs rather than random UUIDs: slightly more complex ID generation, in exchange for idempotent upserts and no stale-point accumulation across re-ingestion runs. Always worth the complexity.
- One collection for all document types rather than one collection per source: simpler query logic — one search endpoint, one filter to scope by source type — rather than needing to know which collection to query for each query type.

### Alternatives & When To Use Each

- One collection per document type — worth considering when different document types need genuinely different vector sizes (different embedding models per type) or when access control requires complete physical separation at the collection level rather than payload-level filtering.
- Payload indexes on all filtered fields — always add these before going to production on any field used frequently in filters; the default of no index is fine for development, not for production at scale.

### Common Mistakes & Production Failures

- Using random point IDs and discovering the collection has accumulated three copies of every chunk after three re-ingestion runs, each slightly stale, all appearing in search results.
- Filtering on a payload field that has no payload index and wondering why filtered queries are slow.
- Not storing the original text in the payload, then needing a secondary lookup to reconstruct chunk text from search results — adds a round trip and a dependency on source files still being present.
- Assuming Qdrant enforces payload schema consistency — it doesn't, and key inconsistency across points only shows up as silent filter misses, not errors.

### Lead-Level Interview Questions

**Q: Why use a hash-based deterministic point ID instead of letting the application generate a random ID?**
A: Random IDs make upsert non-idempotent — re-ingesting a file creates new points rather than replacing old ones, so stale vectors accumulate and appear in search results indefinitely. A deterministic ID (hash of source path + chunk index) means re-upserting the same logical chunk always hits the same ID, updating it in place. This is the vector-store equivalent of the incremental-ingestion manifest's idempotency requirement.

**Q: Qdrant doesn't enforce a payload schema. What can go wrong and how do you prevent it?**
A: Different ingestion code paths can use different key names for the same concept — `"doc_type"` vs `"document_type"` — producing points with inconsistent payload keys. Filters on one key silently miss points using the other, with no error anywhere. Prevention: define the payload schema as an explicit constant that every loader imports and uses, rather than each loader constructing its payload dict independently.

**Q: A collection has 50,000 points. Filtered searches on `doc_type` are slow. What is the most likely cause and fix?**
A: No payload index on `doc_type`. Without an index, Qdrant scans all 50,000 payload records to evaluate the filter during search. Adding a keyword payload index on `doc_type` lets Qdrant resolve the filter via the index instead — operationally identical to adding a database index on a column you filter on frequently.

**Q: What is the difference between `points_count` and `indexed_vectors_count` in Qdrant, and when does the difference matter?**
A: `points_count` is the total number of points stored, including ones still being indexed. `indexed_vectors_count` is the number actually in the HNSW index and searchable. Immediately after a large batch upsert the two can differ — searches during that window may miss recently upserted content. Matters during or immediately after large ingestion batches when the collection is being queried by live traffic concurrently.

### Hidden Concepts Worth Knowing

- **Scroll vs. search**: Qdrant has a `scroll` API for iterating through all points in a collection without a query vector — useful for bulk operations like re-embedding, auditing, or deletion by filter. Worth knowing it exists when you need to, for example, delete all points from a specific source file.
- **Upsert is not a patch**: Qdrant's upsert replaces the entire point if the ID already exists — it does not merge the new payload with the old one. If you upsert a point with a missing payload field, that field is gone from the stored point entirely, not left at its old value.

### Revision Summary

> A collection is a named container for points sharing the same vector size and distance metric. A point is one entry — a unique ID, a vector, and a payload dict. A payload is the metadata that enables filtering and carries the original text for retrieval. Use deterministic hash-based IDs for idempotent upserts, store text in the payload for self-contained results, and add payload indexes on any field used frequently in filters. Qdrant does not enforce payload schema — that is an application-layer responsibility enforced at the loader level.

In [1]:
"""
qdrant_collections_points.py
------------------------------
Demonstrates Qdrant's core data model: collections, points, payloads.

Shows:
  - Creating a collection
  - Generating deterministic point IDs (idempotent upserts)
  - Upserting points with payload mirroring the Document shape
  - Adding a payload index for filtered search performance
  - Retrieving a specific point by ID
  - Scrolling through all points without a query vector
  - Demonstrating idempotent upsert (re-upsert updates, not duplicates)

Uses :memory: mode -- no Docker required.
Install: pip install qdrant-client sentence-transformers
"""

import hashlib

from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance,          # cosine, dot, euclidean -- how to measure vector closeness
    VectorParams,      # vector size + distance metric config for a collection
    PointStruct,       # one point: id + vector + payload
    PayloadSchemaType, # tells Qdrant what type of payload index to create
    Filter,            # wraps filter conditions for a search or scroll
    FieldCondition,    # one condition on a payload field
    MatchValue,        # "must equal this exact value" match type
)
from sentence_transformers import SentenceTransformer

COLLECTION_NAME = "fd_knowledge_base"
VECTOR_SIZE = 384        # must match the embedding model's output size
MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"

# in-memory client -- no Docker, no server needed
client = QdrantClient(":memory:")

# load embedding model once -- reused for every encode() call
model = SentenceTransformer(MODEL_NAME)

# sample documents in the same shape the ingestion chapter produces
DOCUMENTS = [
    {
        "page_content": "Premature withdrawal incurs a 1 percent penalty on the applicable rate.",
        "metadata": {"source": "FD_Policy.json", "doc_type": "policy", "page": 1, "chunk_index": 0},
    },
    {
        "page_content": "This penalty does not apply if the FD is closed due to the death of the depositor.",
        "metadata": {"source": "FD_Policy.json", "doc_type": "policy", "page": 1, "chunk_index": 1},
    },
    {
        "page_content": "Senior citizens receive an additional 0.5 percent interest on all tenures.",
        "metadata": {"source": "FD_Product_Guide.json", "doc_type": "product", "page": 2, "chunk_index": 0},
    },
    {
        "page_content": "What is the minimum amount to open an FD? The minimum deposit is Rs 25,000.",
        "metadata": {"source": "FD_FAQ.json", "doc_type": "faq", "page": 1, "chunk_index": 0},
    },
    {
        "page_content": "Can I withdraw my FD before maturity? Yes, subject to a penalty.",
        "metadata": {"source": "FD_FAQ.json", "doc_type": "faq", "page": 1, "chunk_index": 1},
    },
]


def make_point_id(source: str, chunk_index: int) -> int:
    """Deterministic integer ID from source path + chunk index.
    Same inputs always produce the same ID so re-upserting the same
    chunk updates it in place instead of creating a duplicate point."""
    key = f"{source}::{chunk_index}"
    # take first 8 hex chars of md5 and convert to int -- small enough for Qdrant's uint64 ID
    return int(hashlib.md5(key.encode()).hexdigest()[:8], 16)


def create_collection() -> None:
    # check existing collections to avoid crashing on a duplicate name
    existing = [c.name for c in client.get_collections().collections]

    # delete the old collection if present -- clean slate for this demo
    if COLLECTION_NAME in existing:
        client.delete_collection(COLLECTION_NAME)

    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(
            size=VECTOR_SIZE,          # every point's vector must be exactly this many floats
            distance=Distance.COSINE,  # cosine similarity -- correct for normalized vectors
        ),
    )
    print(f"Created collection '{COLLECTION_NAME}'")


def add_payload_index() -> None:
    """Adds a keyword payload index on 'doc_type'.
    Without this, a filter on doc_type scans every payload record.
    With this, Qdrant resolves the filter via the index -- much faster at scale."""
    client.create_payload_index(
        collection_name=COLLECTION_NAME,
        field_name="doc_type",                  # the payload field to index
        field_schema=PayloadSchemaType.KEYWORD,  # keyword index = exact string match
    )
    print("Added payload index on 'doc_type'")


def upsert_documents(documents: list) -> None:
    # extract text strings for batch embedding
    texts = [d["page_content"] for d in documents]

    # embed all at once -- normalize so dot product == cosine similarity
    embeddings = model.encode(texts, normalize_embeddings=True)

    points = []
    for i, doc in enumerate(documents):
        meta = doc["metadata"]

        # deterministic ID -- same source + chunk_index always gives the same ID
        point_id = make_point_id(meta["source"], meta["chunk_index"])

        points.append(
            PointStruct(
                id=point_id,
                vector=embeddings[i].tolist(),  # numpy array -> plain list for serialization
                payload={
                    "text": doc["page_content"],    # store raw text for self-contained retrieval
                    "source": meta["source"],        # which file this chunk came from
                    "doc_type": meta["doc_type"],    # used for payload filtering
                    "page": meta["page"],            # page within the source document
                    "chunk_index": meta["chunk_index"],  # position within the page
                },
            )
        )

    # upsert = insert if new ID, replace entirely if ID already exists
    client.upsert(collection_name=COLLECTION_NAME, points=points)

    info = client.get_collection(COLLECTION_NAME)
    print(f"Upserted {len(points)} points | Total in collection: {info.points_count}")


def retrieve_by_id(source: str, chunk_index: int) -> None:
    """Fetches one specific point by its deterministic ID -- no query vector needed."""
    point_id = make_point_id(source, chunk_index)

    results = client.retrieve(
        collection_name=COLLECTION_NAME,
        ids=[point_id],
        with_payload=True,   # include the payload fields in the result
        with_vectors=False,  # skip the raw embedding -- not needed for inspection
    )

    if results:
        print(f"\nRetrieved point ID={point_id}:")
        print(f"  text     : {results[0].payload['text'][:80]}")
        print(f"  doc_type : {results[0].payload['doc_type']}")
        print(f"  source   : {results[0].payload['source']}")
    else:
        print(f"No point found for ID={point_id}")


def scroll_all(doc_type_filter: str = None) -> None:
    """Iterates through points without a query vector.
    Used for auditing, bulk re-embedding, or deletion by filter --
    not for similarity search."""

    # build a filter if a doc_type was given, otherwise scroll everything
    scroll_filter = None
    if doc_type_filter:
        scroll_filter = Filter(
            must=[
                FieldCondition(
                    key="doc_type",
                    match=MatchValue(value=doc_type_filter),
                )
            ]
        )

    # scroll returns (results, next_page_offset) -- we only need the results here
    results, _ = client.scroll(
        collection_name=COLLECTION_NAME,
        scroll_filter=scroll_filter,
        with_payload=True,
        with_vectors=False,
        limit=100,  # max points to return in one scroll call
    )

    label = f"doc_type='{doc_type_filter}'" if doc_type_filter else "all"
    print(f"\nScroll ({label}): {len(results)} points")
    for r in results:
        print(f"  [{r.payload['doc_type']}] {r.payload['text'][:60]}")


def demonstrate_idempotent_upsert() -> None:
    """Re-upserts a point that already exists (same deterministic ID).
    Point count stays the same -- it updated in place, not duplicated."""

    # record the count before re-upserting
    count_before = client.get_collection(COLLECTION_NAME).points_count

    # same source + chunk_index as the first document -- same ID will be generated
    updated_doc = {
        "page_content": "UPDATED: Premature withdrawal penalty is 1 percent.",
        "metadata": {
            "source": "FD_Policy.json",
            "doc_type": "policy",
            "page": 1,
            "chunk_index": 0,  # same chunk_index -> same ID -> in-place update
        },
    }
    upsert_documents([updated_doc])

    count_after = client.get_collection(COLLECTION_NAME).points_count

    print(f"\nIdempotent upsert: before={count_before}, after={count_after}")
    print("  Count unchanged -- updated in place, no duplicate created")

    # confirm the content was actually updated
    retrieve_by_id("FD_Policy.json", chunk_index=0)


# ----------------------------------------------------------------------
# Run all demos in order
# ----------------------------------------------------------------------

# step 1: create collection
create_collection()

# step 2: add payload index BEFORE upserting so Qdrant indexes as points arrive
add_payload_index()

# step 3: embed and upsert all sample documents
upsert_documents(DOCUMENTS)

# step 4: fetch one specific point by its deterministic ID
retrieve_by_id("FD_Policy.json", chunk_index=0)

# step 5: scroll through all points (no query vector)
scroll_all()

# step 6: scroll only FAQ points using the payload index
scroll_all(doc_type_filter="faq")

# step 7: re-upsert one point and show the count stays the same
demonstrate_idempotent_upsert()


Created collection 'fd_knowledge_base'
Added payload index on 'doc_type'
Upserted 5 points | Total in collection: 5

Retrieved point ID=3267135047:
  text     : Premature withdrawal incurs a 1 percent penalty on the applicable rate.
  doc_type : policy
  source   : FD_Policy.json

Scroll (all): 5 points
  [faq] What is the minimum amount to open an FD? The minimum deposi
  [faq] Can I withdraw my FD before maturity? Yes, subject to a pena
  [product] Senior citizens receive an additional 0.5 percent interest o
  [policy] This penalty does not apply if the FD is closed due to the d
  [policy] Premature withdrawal incurs a 1 percent penalty on the appli

Scroll (doc_type='faq'): 2 points
  [faq] What is the minimum amount to open an FD? The minimum deposi
  [faq] Can I withdraw my FD before maturity? Yes, subject to a pena
Upserted 1 points | Total in collection: 5

Idempotent upsert: before=5, after=5
  Count unchanged -- updated in place, no duplicate created

Retrieved point ID=32671

C:\Users\pauls\AppData\Local\Temp\ipykernel_42232\3686356635.py:99: UserWarning: Payload indexes have no effect in the local Qdrant. Please use server Qdrant if you need payload indexes.
  client.create_payload_index(
